# LeetCode 1856 — Maximum Subarray Min-Product bằng Python

Link đề: https://leetcode.com/problems/maximum-subarray-min-product/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Monotonic Stack, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maxSumMinProduct(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đổi góc nhìn "duyệt mọi mảng con" thành "**coi mỗi phần tử là giá trị nhỏ nhất**": với min cố định `nums[i]`, mảng con tốt nhất là đoạn **rộng nhất** quanh `i` mà mọi phần tử `≥ nums[i]` (vì mọi số đều dương).
2. Tìm "phần tử nhỏ hơn gần nhất bên trái/phải" bằng **stack đơn điệu** trong `O(n)`.
3. Tính tổng đoạn `O(1)` bằng **tổng tiền tố**.
4. Xử lý **phần tử bằng nhau** (một phía `<`, phía kia `≤`) và chỉ lấy **modulo sau khi** tìm max.
5. Thêm hai cách nhìn khác: tính ngay khi **pop** khỏi stack (một lượt), và **DSU** kích hoạt phần tử theo thứ tự giảm dần.

## 2. Đề bài & ví dụ

**Min-product** của một mảng = (giá trị **nhỏ nhất** của mảng) × (**tổng** mảng). Ví dụ `[3,2,5]` có min-product `2 · (3+2+5) = 20`.

Cho mảng số nguyên `nums`, trả về **min-product lớn nhất** trong mọi **mảng con khác rỗng** (liên tiếp), theo modulo `10⁹ + 7`. Lưu ý: phải **tối đa hóa trước** rồi mới lấy modulo. Đề đảm bảo giá trị lớn nhất (chưa mod) vừa số nguyên 64-bit có dấu.

**Ví dụ 1:** `nums = [1,2,3,2]` → `14` (`[2,3,2]`: `2 · 7`).

**Ví dụ 2:** `nums = [2,3,3,1,2]` → `18` (`[3,3]`: `3 · 6`).

**Ví dụ 3:** `nums = [3,1,5,6,4,2]` → `60` (`[5,6,4]`: `4 · 15`).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `1 ≤ nums[i] ≤ 10⁷`.

In [1]:
import random
import time
from itertools import accumulate
from typing import List

MOD = 10 ** 9 + 7
examples = [([1, 2, 3, 2], 14), ([2, 3, 3, 1, 2], 18), ([3, 1, 5, 6, 4, 2], 60)]
for nums, exp in examples:
    print(f"nums={nums} -> kỳ vọng {exp}")

nums=[1, 2, 3, 2] -> kỳ vọng 14
nums=[2, 3, 3, 1, 2] -> kỳ vọng 18
nums=[3, 1, 5, 6, 4, 2] -> kỳ vọng 60


## 3. Kiến thức nền

### 3.1 Tổng tiền tố

`pre[i] = nums[0] + … + nums[i−1]` ⇒ tổng `nums[l..r] = pre[r+1] − pre[l]`.

In [2]:
nums = [3, 1, 5, 6, 4, 2]
pre = list(accumulate(nums, initial=0))
print("pre =", pre, "| tổng nums[2..4] =", pre[5] - pre[2])

pre = [0, 3, 4, 9, 15, 19, 21] | tổng nums[2..4] = 15


### 3.2 Stack đơn điệu: phần tử nhỏ hơn gần nhất

Duyệt trái → phải, giữ stack các chỉ số có giá trị **tăng dần**. Khi gặp `x`, pop mọi phần tử `≥ x`; đỉnh còn lại (nếu có) là chỉ số gần nhất bên trái có giá trị `< x`. Mỗi chỉ số push/pop một lần ⇒ `O(n)`.

In [3]:
def prev_smaller(nums):
    left, st = [-1] * len(nums), []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] >= x:
            st.pop()
        left[i] = st[-1] if st else -1
        st.append(i)
    return left

print(prev_smaller([3, 1, 5, 6, 4, 2]))

[-1, -1, 1, 2, 1, 1]


### 3.3 Modulo sau khi tối đa

`max(a % M, b % M)` **khác** `max(a, b) % M`. Phải so sánh giá trị thật (Python có số nguyên lớn), cuối cùng mới `% M`.

In [4]:
a, b = MOD + 5, 100
print("max rồi mod:", max(a, b) % MOD, "| mod rồi max:", max(a % MOD, b % MOD))

max rồi mod: 5 | mod rồi max: 100


### 3.4 Disjoint Set Union (DSU) với tổng đoạn

Mỗi tập lưu `sum`; `union` gộp hai đoạn kề nhau và cộng tổng. Dùng ở Cách 5.

### 3.5 Vì sao điều kiện "số dương" quan trọng

Lập luận "mở rộng tối đa chỉ làm tổng tăng" dựa vào việc mọi phần tử `≥ 1`. Nếu có số âm hoặc 0, mở rộng có thể làm tổng **giảm** — khi đó đoạn tối đa chưa chắc tốt nhất. Thí nghiệm nhỏ dưới đây (dữ liệu **ngoài ràng buộc đề**, chỉ để minh họa) cho thấy cách "đoạn tối đa" sai khi có số âm.

In [5]:
def brute_any(nums):
    return max(min(nums[i:j]) * sum(nums[i:j]) for i in range(len(nums)) for j in range(i + 1, len(nums) + 1))

def widest_any(nums):
    n = len(nums); L = prev_smaller(nums)
    R = [n - 1 - x for x in prev_smaller(nums[::-1])][::-1]
    return max(nums[i] * sum(nums[L[i] + 1:R[i]]) for i in range(n))

arr = [3, -5, 4]          # ngoài ràng buộc (có số âm)
print("vét cạn:", brute_any(arr), "| đoạn tối đa:", widest_any(arr))
rng = random.Random(35)
pos_ok = all(brute_any(a) == widest_any(a) for a in ([rng.randint(1, 9) for _ in range(rng.randint(1, 8))] for _ in range(300)))
print("Với số dương, hai cách luôn trùng (300 test):", pos_ok)

vét cạn: 25 | đoạn tối đa: 16
Với số dương, hai cách luôn trùng (300 test): True


## 4. Ý tưởng nền tảng / trực giác

Mọi mảng con có một phần tử nhỏ nhất. Cố định "phần tử nhỏ nhất là `nums[i]`": mảng con phải nằm trong đoạn `(L, R)` với `L` = chỉ số gần nhất bên trái có giá trị **nhỏ hơn**, `R` = gần nhất bên phải có giá trị **nhỏ hơn**. Vì mọi số **dương**, mở rộng tối đa trong `(L, R)` chỉ làm **tổng tăng** mà min không đổi ⇒ chọn cả đoạn `[L+1, R−1]`.

Tính tay **Ví dụ 3** `[3,1,5,6,4,2]`:

| i | nums[i] | L | R | đoạn | tổng | min-product |
|---|---|---|---|---|---|---|
| 0 | 3 | −1 | 1 | [3] | 3 | 9 |
| 1 | 1 | −1 | 6 | cả mảng | 21 | 21 |
| 2 | 5 | 1 | 4 | [5,6] | 11 | 55 |
| 3 | 6 | 2 | 4 | [6] | 6 | 36 |
| 4 | 4 | 1 | 5 | [5,6,4] | 15 | **60** |
| 5 | 2 | 1 | 6 | [5,6,4,2] | 17 | 34 |

In [6]:
nums = [3, 1, 5, 6, 4, 2]; n = len(nums)
for i in range(n):
    L = i - 1
    while L >= 0 and nums[L] >= nums[i]: L -= 1
    R = i + 1
    while R < n and nums[R] >= nums[i]: R += 1
    seg = nums[L + 1:R]
    print(f"i={i} ({nums[i]}): đoạn {seg}, min-product {nums[i] * sum(seg)}")

i=0 (3): đoạn [3], min-product 9
i=1 (1): đoạn [3, 1, 5, 6, 4, 2], min-product 21
i=2 (5): đoạn [5, 6], min-product 55
i=3 (6): đoạn [6], min-product 36
i=4 (4): đoạn [5, 6, 4], min-product 60
i=5 (2): đoạn [5, 6, 4, 2], min-product 34


## 5. Cách 1 — Brute Force / baseline: mọi mảng con, giữ min và tổng chạy

Cố định đầu `i`, mở rộng `j`, cập nhật `mn = min(mn, nums[j])`, `s += nums[j]` — `O(1)` mỗi mảng con.

In [7]:
def minprod_bruteforce(nums):
    best = 0
    for i in range(len(nums)):
        mn, s = float("inf"), 0
        for j in range(i, len(nums)):
            mn = min(mn, nums[j]); s += nums[j]
            best = max(best, mn * s)
    return best % MOD

for nums, exp in examples:
    got = minprod_bruteforce(nums); print(got, "kỳ vọng", exp); assert got == exp

14 kỳ vọng 14
18 kỳ vọng 18
60 kỳ vọng 60


**Độ phức tạp:** `O(n²)` = `5·10⁹` với `n = 10⁵` ⇒ **TLE**.

### Cách 1' — Với mỗi `i`, mở rộng hai phía tuyến tính (như mục 4)

`O(n²)` tệ nhất (mảng toàn bằng nhau: mỗi `i` mở rộng tới hết). Ý tưởng đúng, chỉ cần tăng tốc việc tìm `L`, `R`.

In [8]:
def minprod_expand(nums):
    n = len(nums); pre = list(accumulate(nums, initial=0)); best = 0
    for i in range(n):
        L = i - 1
        while L >= 0 and nums[L] >= nums[i]: L -= 1
        R = i + 1
        while R < n and nums[R] >= nums[i]: R += 1
        best = max(best, nums[i] * (pre[R] - pre[L + 1]))
    return best % MOD

print([minprod_expand(n) for n, _ in examples])

[14, 18, 60]


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (phần tử làm min):** đáp án `= max_i nums[i] · sum(L_i + 1 .. R_i − 1)`, với `L_i`, `R_i` là phần tử **nhỏ hơn** gần nhất hai bên.

**Insight 2 (stack đơn điệu):** `L_i`, `R_i` cho mọi `i` trong `O(n)`.

**Insight 3 (phần tử bằng nhau):** nếu dùng `<` nghiêm ngặt **cả hai phía**, các phần tử bằng nhau mở rộng qua nhau ⇒ vẫn đúng (mỗi bản đều tìm được đoạn tối đa) — vì ta lấy **max**, không cần tránh đếm trùng. (Bài **đếm/tổng** trên mọi mảng con thì mới cần một phía `≤`.)

**Quy tắc stack (trái):** "trong khi đỉnh `≥ x` thì pop" ⇒ đỉnh còn lại `< x`.

**Invariant:** stack chứa các chỉ số có giá trị **tăng nghiêm ngặt** từ đáy lên đỉnh.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai lượt stack (trái, phải) + tổng tiền tố

In [9]:
def minprod_two_stacks(nums):
    n = len(nums); left, right, st = [-1] * n, [n] * n, []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] >= x: st.pop()
        left[i] = st[-1] if st else -1
        st.append(i)
    st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] >= nums[i]: st.pop()
        right[i] = st[-1] if st else n
        st.append(i)
    pre = list(accumulate(nums, initial=0))
    return max(nums[i] * (pre[right[i]] - pre[left[i] + 1]) for i in range(n)) % MOD

print([minprod_two_stacks(n) for n, _ in examples])

[14, 18, 60]


**Code nộp LeetCode (Cách 2):** xem mục 9. **Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### 7.2 Cách 3 — Một lượt: tính khi pop

Khi chỉ số `j` bị pop bởi `i` (điều kiện pop: `nums[j] > nums[i]`), `i` chính là `R_j` — phần tử nhỏ hơn đầu tiên bên phải; phần tử nằm dưới `j` trong stack là `L_j`. Tính min-product của `j` ngay lúc pop; cuối cùng pop nốt với `R = n`.

In [10]:
class SolutionOnePass:
    def maxSumMinProduct(self, nums: List[int]) -> int:
        n = len(nums)
        pre = list(accumulate(nums, initial=0))
        st = []
        best = 0
        for i in range(n + 1):
            cur = nums[i] if i < n else 0                 # phần tử "lính canh" 0 ở cuối để pop hết
            while st and nums[st[-1]] > cur:
                j = st.pop()
                L = st[-1] if st else -1
                best = max(best, nums[j] * (pre[i] - pre[L + 1]))
            st.append(i)
        return best % (10 ** 9 + 7)

print([SolutionOnePass().maxSumMinProduct(n) for n, _ in examples])

[14, 18, 60]


Lưu ý: pop với `>` (nghiêm ngặt) nên các phần tử **bằng nhau** vẫn ở lại stack; phần tử bằng nhau ở dưới `j` thì `L` của `j` "dừng sớm" tại nó — đoạn của `j` bị thiếu một phía. Nhưng bản sao **cuối cùng** (bị pop sau cùng) sẽ có `L` đúng là phần tử nhỏ hơn thật sự, nên max vẫn đúng.

**Độ phức tạp Cách 3:** `O(n)`, một lượt.

### 7.3 Cách 4 — DSU: kích hoạt phần tử theo thứ tự giảm dần

Duyệt chỉ số theo giá trị **giảm dần**. Kích hoạt `i`, gộp với hàng xóm trái/phải nếu đã kích hoạt (chúng `≥ nums[i]`). Lúc vừa gộp xong, đoạn chứa `i` là đoạn tối đa có mọi phần tử `≥ nums[i]` ⇒ ứng viên `nums[i] · tổng đoạn`.

In [11]:
def minprod_dsu(nums):
    n = len(nums); parent = list(range(n)); seg = [0] * n; active = [False] * n; best = 0
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra; seg[ra] += seg[rb]
        return ra
    for i in sorted(range(n), key=lambda t: -nums[t]):
        active[i] = True; seg[i] = nums[i]; r = i
        if i > 0 and active[i - 1]: r = union(i - 1, i)
        if i + 1 < n and active[i + 1]: r = union(r, i + 1)
        best = max(best, nums[i] * seg[find(i)])
    return best % MOD

print([minprod_dsu(n) for n, _ in examples])

[14, 18, 60]


**Độ phức tạp Cách 4:** sắp `O(n log n)` + DSU gần `O(n)` ⇒ `O(n log n)`. Kỹ thuật "kích hoạt theo thứ tự + DSU" dùng lại cho nhiều bài (ví dụ 2334, 1562).

## 8. TRACE từng bước

Trace Cách 3 (một lượt) trên **Ví dụ 2** `[2,3,3,1,2]`:

In [12]:
def trace_one_pass(nums):
    n = len(nums); pre = list(accumulate(nums, initial=0)); st = []; best = 0
    for i in range(n + 1):
        cur = nums[i] if i < n else 0
        while st and nums[st[-1]] > cur:
            j = st.pop(); L = st[-1] if st else -1
            val = nums[j] * (pre[i] - pre[L + 1]); best = max(best, val)
            print(f"  i={i}: pop j={j} (nums={nums[j]}), đoạn [{L + 1}..{i - 1}] = {nums[L + 1:i]}, min-product {val}, best={best}")
        st.append(i)
        print(f"i={i}: push -> stack giá trị {[nums[t] if t < n else 0 for t in st]}")
    return best % MOD

assert trace_one_pass([2, 3, 3, 1, 2]) == 18

i=0: push -> stack giá trị [2]
i=1: push -> stack giá trị [2, 3]
i=2: push -> stack giá trị [2, 3, 3]
  i=3: pop j=2 (nums=3), đoạn [2..2] = [3], min-product 9, best=9
  i=3: pop j=1 (nums=3), đoạn [1..2] = [3, 3], min-product 18, best=18
  i=3: pop j=0 (nums=2), đoạn [0..2] = [2, 3, 3], min-product 16, best=18
i=3: push -> stack giá trị [1]
i=4: push -> stack giá trị [1, 2]
  i=5: pop j=4 (nums=2), đoạn [4..4] = [2], min-product 4, best=18
  i=5: pop j=3 (nums=1), đoạn [0..4] = [2, 3, 3, 1, 2], min-product 11, best=18
i=5: push -> stack giá trị [0]


| sự kiện | j bị pop | đoạn | min-product |
|---|---|---|---|
| i=3 (giá trị 1) | 2 (3) | [2..2] = [3] | 9 |
| | 1 (3) | [1..2] = [3,3] | **18** |
| | 0 (2) | [0..2] = [2,3,3] | 16 |
| i=5 (lính canh) | 4 (2) | [4..4] | 4 |
| | 3 (1) | [0..4] | 11 |

`j = 2` bị pop trước; phần tử dưới nó là `j = 1` có **cùng** giá trị 3 nên đoạn của `j = 2` chỉ là `[3]` (9). Đoạn đầy đủ `[3,3]` được ghi nhận ở lần pop `j = 1` — bản sao thấp nhất trong stack luôn có biên trái đúng.

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — hai lượt stack đơn điệu + tổng tiền tố.

In [13]:
class Solution:
    def maxSumMinProduct(self, nums: List[int]) -> int:
        n = len(nums)
        left = [-1] * n
        right = [n] * n
        st = []
        for i, x in enumerate(nums):
            while st and nums[st[-1]] >= x:
                st.pop()
            if st:
                left[i] = st[-1]
            st.append(i)
        st = []
        for i in range(n - 1, -1, -1):
            while st and nums[st[-1]] >= nums[i]:
                st.pop()
            if st:
                right[i] = st[-1]
            st.append(i)
        pre = list(accumulate(nums, initial=0))
        best = 0
        for i in range(n):
            best = max(best, nums[i] * (pre[right[i]] - pre[left[i] + 1]))
        return best % (10 ** 9 + 7)

for nums, exp in examples:
    got = Solution().maxSumMinProduct(nums); print(got, exp); assert got == exp

14 14
18 18
60 60


## 10. Vì sao thuật toán đúng

**Cận trên.** Xét mảng con tối ưu `S = nums[l..r]` với min `m = nums[i]` (một chỉ số đạt min). Mọi phần tử trong `S` `≥ m` ⇒ `S ⊆ [L_i + 1, R_i − 1]` (vì `L_i`, `R_i` là phần tử **nhỏ hơn** `m` gần nhất, không thể nằm trong `S`). Đoạn `[L_i+1, R_i−1]` có min đúng bằng `m` (mọi phần tử `≥ m`, và chứa `i`) và tổng `≥` tổng của `S` (số dương). Vậy ứng viên của `i` `≥` min-product của `S`.

**Mỗi ứng viên là min-product của một mảng con thật** ⇒ không vượt tối ưu. Do đó max các ứng viên = tối ưu. ∎

(Với Cách 2, phía trái dùng `≥` để pop nghĩa là `L_i` là phần tử `< nums[i]` gần nhất; phía phải cũng vậy — cả hai nghiêm ngặt, đúng định nghĩa ở trên.)

### 10.1 Kiểm chứng thực nghiệm hai mệnh đề trong chứng minh

(a) Với mỗi mảng con tối ưu tìm bằng vét cạn, tồn tại `i` trong nó đạt min sao cho mảng con nằm trong `[L_i+1, R_i−1]`. (b) Mỗi ứng viên `nums[i]·sum(L_i+1..R_i−1)` đúng bằng min-product thật của đoạn đó.

In [14]:
def check_claims(nums):
    n = len(nums); L = prev_smaller(nums); R = next_smaller_todo_ref(nums)
    for i in range(n):                                   # (b)
        seg = nums[L[i] + 1:R[i]]
        assert min(seg) == nums[i]
    best = max((min(nums[a:b]) * sum(nums[a:b]), a, b - 1) for a in range(n) for b in range(a + 1, n + 1))
    v, l, r = best
    assert any(nums[i] == min(nums[l:r + 1]) and L[i] < l and r < R[i] for i in range(l, r + 1))   # (a)

def next_smaller_todo_ref(nums):
    n = len(nums)
    return [n - 1 - x for x in prev_smaller(nums[::-1])][::-1]

rng = random.Random(101)
for _ in range(400):
    check_claims([rng.randint(1, 5) for _ in range(rng.randint(1, 10))])
print("Hai mệnh đề (a), (b) đúng trên 400 mảng ngẫu nhiên")

Hai mệnh đề (a), (b) đúng trên 400 mảng ngẫu nhiên


## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi mảng con | `O(n²)` | `O(1)` |
| 1'. Mở rộng hai phía | `O(n²)` tệ nhất | `O(n)` |
| 2. Hai stack + tiền tố | `O(n)` | `O(n)` |
| 3. Một lượt pop | `O(n)` | `O(n)` |
| 4. DSU theo thứ tự giảm | `O(n log n)` | `O(n)` |

Độ lớn: tổng đoạn `≤ 10⁵ · 10⁷ = 10¹²`, nhân min `≤ 10⁷` ⇒ tích `≤ 10¹⁹` về lý thuyết; đề đảm bảo đáp án thật vừa `int64`. Python dùng số nguyên lớn nên không lo tràn; trong C++/Java cần `long long`/`long`.

In [15]:
rng = random.Random(1856)
big = [rng.randint(1, 10 ** 7) for _ in range(10 ** 5)]
same = [7] * 10 ** 5
for name, f in [("two_stacks", Solution().maxSumMinProduct), ("one_pass", SolutionOnePass().maxSumMinProduct), ("dsu", minprod_dsu)]:
    for label, data in [("ngẫu nhiên", big), ("toàn bằng nhau", same)]:
        t = time.perf_counter(); r = f(data); print(f"{name:<10} {label:<15}: {r:>10} ({(time.perf_counter()-t)*1000:.0f} ms)")
t = time.perf_counter(); minprod_expand([7] * 2000); print(f"Cách 1' toàn bằng nhau n=2000: {(time.perf_counter()-t)*1000:.0f} ms")

two_stacks ngẫu nhiên     :  873853298 (34 ms)
two_stacks toàn bằng nhau :    4900000 (21 ms)
one_pass   ngẫu nhiên     :  873853298 (20 ms)
one_pass   toàn bằng nhau :    4900000 (17 ms)


dsu        ngẫu nhiên     :  873853298 (84 ms)
dsu        toàn bằng nhau :    4900000 (45 ms)


Cách 1' toàn bằng nhau n=2000: 126 ms


## 12. Bộ test

In [16]:
ALL = {"expand": minprod_expand, "two_stacks": minprod_two_stacks, "one_pass": SolutionOnePass().maxSumMinProduct,
       "dsu": minprod_dsu, "submit": Solution().maxSumMinProduct}
def run_all(nums, brute=True):
    res = {k: f(list(nums)) for k, f in ALL.items()}
    if brute:
        res["brute"] = minprod_bruteforce(nums)
    assert len(set(res.values())) == 1, (nums, res)
    return res["submit"]

for nums, exp in examples:
    assert run_all(nums) == exp
edge = [([5], 25), ([1, 1, 1], 3), ([10 ** 7], (10 ** 14) % MOD), ([1, 100], 10000), ([3, 3, 1, 3, 3], 18), ([2, 1, 2], 5)]
for nums, exp in edge:
    assert run_all(nums) == exp, nums
rng = random.Random(18560)
for _ in range(600):
    run_all([rng.randint(1, rng.choice([3, 10, 10 ** 7])) for _ in range(rng.randint(1, 40))])
for _ in range(5):
    big = [rng.randint(1, 10 ** 7) for _ in range(3000)]
    assert len({f(big) for f in (minprod_two_stacks, SolutionOnePass().maxSumMinProduct, minprod_dsu)}) == 1
big = [10 ** 7] * 10 ** 5
assert Solution().maxSumMinProduct(big) == (10 ** 7 * 10 ** 12) % MOD
print("Ví dụ, edge case, 600 test vét cạn, 5 test vừa, test lớn (mod sau max): OK")

Ví dụ, edge case, 600 test vét cạn, 5 test vừa, test lớn (mod sau max): OK


### 12.1 Kiểm tra tính chất (property tests)

- Hoán vị ngược mảng (`nums[::-1]`) không đổi đáp án (mảng con ngược vẫn là mảng con).
- Nhân mọi phần tử với `k` ⇒ đáp án (chưa mod) nhân `k²`.
- Thêm một phần tử vào cuối không làm đáp án (chưa mod) giảm.

In [17]:
def raw_best(nums):
    n = len(nums); L = prev_smaller(nums); R = next_smaller_todo_ref(nums); pre = list(accumulate(nums, initial=0))
    return max(nums[i] * (pre[R[i]] - pre[L[i] + 1]) for i in range(n))

rng = random.Random(121)
for _ in range(500):
    a = [rng.randint(1, 50) for _ in range(rng.randint(1, 30))]; k = rng.randint(2, 5)
    assert raw_best(a) == raw_best(a[::-1])
    assert raw_best([k * x for x in a]) == k * k * raw_best(a)
    assert raw_best(a + [rng.randint(1, 50)]) >= raw_best(a)
print("3 tính chất đúng trên 500 mảng")

3 tính chất đúng trên 500 mảng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lấy modulo **trước** khi so sánh

In [18]:
def wrong_mod_early(nums):
    n = len(nums); pre = list(accumulate(nums, initial=0)); best = 0
    left = prev_smaller(nums)
    right = [n - 1 - x for x in prev_smaller(nums[::-1])][::-1]
    for i in range(n):
        best = max(best, (nums[i] * (pre[right[i]] - pre[left[i] + 1])) % MOD)     # SAI
    return best

nums = [1, 31623]          # ví dụ tự dựng (đối chiếu lại đề gốc): 31623² = 1000014129 > MOD
print("ứng viên thật:", 1 * 31624, "và", 31623 ** 2, "| 31623² % MOD =", 31623 ** 2 % MOD)
print("sai :", wrong_mod_early(nums), "| đúng:", Solution().maxSumMinProduct(nums), "| brute:", minprod_bruteforce(nums))
assert wrong_mod_early(nums) != Solution().maxSumMinProduct(nums) == minprod_bruteforce(nums) == 14122

ứng viên thật: 31624 và 1000014129 | 31623² % MOD = 14122
sai : 31624 | đúng: 14122 | brute: 14122


**Vì sao sai:** sau khi lấy mod, số lớn có thể trở thành nhỏ và thua một ứng viên thật sự nhỏ hơn. Đề nói rõ: tối đa hóa **trước**, mod **sau**. **Sửa:** so sánh giá trị thật, `% MOD` ở cuối.

### Lỗi 2 — Chỉ xét mảng con kết thúc/bắt đầu tại phần tử min (không mở rộng hết)

In [19]:
def wrong_one_side(nums):
    n = len(nums); pre = list(accumulate(nums, initial=0)); best = 0
    left = prev_smaller(nums)
    for i in range(n):
        best = max(best, nums[i] * (pre[i + 1] - pre[left[i] + 1]))   # SAI: chỉ mở rộng sang trái
    return best % MOD

nums = [5, 6, 1]          # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_one_side(nums), "| đúng:", Solution().maxSumMinProduct(nums), "| brute:", minprod_bruteforce(nums))
print("Ví dụ 3 vẫn trùng (may mắn):", wrong_one_side(examples[2][0]), "— vì đoạn tối ưu [5,6,4] kết thúc đúng tại min 4")

sai : 36 | đúng: 55 | brute: 55
Ví dụ 3 vẫn trùng (may mắn): 60 — vì đoạn tối ưu [5,6,4] kết thúc đúng tại min 4


**Vì sao sai:** với `[5,6,1]`, đoạn tối ưu là `[5,6]` = `5·11 = 55`, trong đó min `5` nằm **đầu** đoạn — chỉ mở sang trái thì không bao giờ thấy nó (bản sai cho `6·6 = 36`). Ví dụ 3 chỉ trùng do đoạn tối ưu tình cờ kết thúc tại min. Phải mở **cả hai phía**. **Sửa:** dùng cả `left` và `right`.

### Lỗi 3 — Chọn đoạn có **tổng lớn nhất** rồi nhân min (tham lam sai)

In [20]:
def wrong_max_sum(nums):
    return (min(nums) * sum(nums)) % MOD                     # SAI: cả mảng

for nums, exp in examples:
    print("sai :", wrong_max_sum(nums), "| đúng:", exp)

sai : 8 | đúng: 14
sai : 11 | đúng: 18
sai : 21 | đúng: 60


**Vì sao sai:** tổng lớn nhất là cả mảng nhưng min có thể rất nhỏ (ví dụ có số 1). Đánh đổi giữa min và tổng — phải thử mỗi phần tử làm min. **Sửa:** Cách 2.

### Lỗi 4 — Stack pop sai chiều (`<` thay vì `≥`) ⇒ tìm phần tử **lớn hơn** gần nhất

In [21]:
def wrong_stack_dir(nums):
    n = len(nums); left, right, st = [-1] * n, [n] * n, []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] < x: st.pop()              # SAI chiều
        left[i] = st[-1] if st else -1; st.append(i)
    st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] < nums[i]: st.pop()
        right[i] = st[-1] if st else n; st.append(i)
    pre = list(accumulate(nums, initial=0))
    return max(nums[i] * (pre[right[i]] - pre[left[i] + 1]) for i in range(n)) % MOD

nums, exp = examples[0]
print("sai :", wrong_stack_dir(nums), "| đúng:", exp)

sai : 24 | đúng: 14


**Vì sao sai:** đoạn tìm được là nơi `nums[i]` là **max**, không phải min. **Sửa:** pop khi `nums[st[-1]] ≥ x` (giữ stack tăng dần).

### Lỗi 5 — Tính tổng đoạn bằng `sum(nums[L+1:R])` trong vòng lặp (`O(n²)`)

In [22]:
def slow_sum(nums):
    n = len(nums); left = prev_smaller(nums)
    right = [n - 1 - x for x in prev_smaller(nums[::-1])][::-1]
    return max(nums[i] * sum(nums[left[i] + 1:right[i]]) for i in range(n)) % MOD

arr = [5] * 4000
for f in (slow_sum, Solution().maxSumMinProduct):
    t = time.perf_counter(); r = f(arr); print(f"{getattr(f, '__name__', '?'):<17} -> {r} ({(time.perf_counter()-t)*1000:.0f} ms)")

slow_sum          -> 100000 (68 ms)
maxSumMinProduct  -> 100000 (1 ms)


**Vì sao không tốt:** mảng toàn bằng nhau ⇒ mỗi đoạn dài `n` ⇒ `O(n²)`. **Sửa:** tổng tiền tố.

### Lỗi 6 — Lệch chỉ số khi dùng tổng tiền tố

Viết `pre[right[i]] - pre[left[i]]` (quên `+1`) ⇒ tổng bao gồm cả phần tử `nums[left[i]]` — phần tử nhỏ hơn nằm **ngoài** đoạn.

In [23]:
def wrong_prefix(nums):
    n = len(nums); L = prev_smaller(nums); R = next_smaller_todo_ref(nums); pre = list(accumulate(nums, initial=0))
    return max(nums[i] * (pre[R[i]] - pre[max(L[i], 0)]) for i in range(n)) % MOD     # SAI

for nums, exp in examples:
    print("sai :", wrong_prefix(nums), "| đúng:", exp)

sai : 16 | đúng: 14
sai : 24 | đúng: 18
sai : 66 | đúng: 60


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối ưu một biểu thức chứa **min (hoặc max) của mảng con**" ⇒ **cố định phần tử làm min**, tìm biên bằng **stack đơn điệu** (giống 84 Largest Rectangle in Histogram).
- Số dương ⇒ mở rộng đoạn chỉ có lợi cho tổng ⇒ lấy đoạn tối đa.
- Tổng đoạn ⇒ tổng tiền tố.
- Có modulo nhưng phải "tối đa trước" ⇒ giữ số thật (Python) hoặc `long` (C++/Java).
- Phần tử bằng nhau: với bài **max** thì không lo đếm trùng; với bài **tổng/đếm** cần một phía `<`, một phía `≤`.

## 15. Biến thể của bài

1. **84 Largest Rectangle in Histogram**: `nums[i] · (R − L − 1)` thay cho tổng — cùng khung.
2. **907 Sum of Subarray Minimums**: **tổng** min của mọi mảng con — đóng góp `nums[i] · (i − L) · (R − i)` với một phía `≤` (Bài tập 5).
3. **2104 Sum of Subarray Ranges**: tổng (max − min) — hai lần 907.
4. **Max (min × độ dài)** hoặc **max (max × tổng)** — đổi vai stack.
5. **Có số âm**: mở rộng không còn luôn có lợi — bài khó hơn hẳn.

### 15.1 Code biến thể: Largest Rectangle in Histogram (84) cùng khung

In [24]:
def largest_rectangle(heights):
    n = len(heights); L = prev_smaller(heights); R = next_smaller_todo_ref(heights)
    return max(heights[i] * (R[i] - L[i] - 1) for i in range(n))

def largest_rectangle_brute(h):
    return max(min(h[i:j]) * (j - i) for i in range(len(h)) for j in range(i + 1, len(h) + 1))

print(largest_rectangle([2, 1, 5, 6, 2, 3]), "(LeetCode 84: 10)")
rng = random.Random(84)
for _ in range(300):
    h = [rng.randint(1, 8) for _ in range(rng.randint(1, 10))]
    assert largest_rectangle(h) == largest_rectangle_brute(h)
print("84 khớp vét cạn trên 300 test — chỉ đổi 'tổng' thành 'độ rộng'")

10 (LeetCode 84: 10)
84 khớp vét cạn trên 300 test — chỉ đổi 'tổng' thành 'độ rộng'


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi mảng con | Min, tổng chạy | `O(n²)` | `O(1)` | Kiểm chứng |
| 1'. Mở rộng | Mỗi phần tử làm min, quét hai phía | `O(n²)` | `O(n)` | Hiểu ý tưởng |
| 2. Hai stack | Phần tử nhỏ hơn gần nhất | `O(n)` | `O(n)` | **Bản nộp** |
| 3. Một lượt pop | Tính khi pop, lính canh 0 | `O(n)` | `O(n)` | Gọn, một lượt |
| 4. DSU giảm dần | Kích hoạt + gộp đoạn | `O(n log n)` | `O(n)` | Khuôn mẫu "kích hoạt theo ngưỡng" |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[1]`; (b) `[2,2]`; (c) `[1,5,1]`; (d) `[4,4,1,4,4,4]`.

**Bài 2 (tính tay):** Lập bảng `L`, `R`, đoạn, min-product cho `[2,5,4,1,3]`.

**Bài 3 (điền TODO):** Hoàn thiện `next_smaller(nums)` trả mảng `R` (phần tử nhỏ hơn gần nhất bên phải, `n` nếu không có).

**Bài 4 (tự viết lại):** Viết `best_subarray(nums)` trả `(min-product chưa mod, l, r)` của **một** mảng con tối ưu.

**Bài 5 (mini challenge):** Viết `sum_subarray_mins(arr)` (LeetCode 907): tổng `min` của mọi mảng con, mod `10⁹ + 7`, `O(n)` bằng stack; đối chiếu vét cạn. Chú ý phần tử bằng nhau.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def next_smaller_todo(nums):
    n = len(nums); right = [n] * n; st = []
    for i in range(n - 1, -1, -1):
        # TODO: pop khi nums[st[-1]] >= nums[i]; right[i] = đỉnh nếu có; push i
        pass
    return right

In [26]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def best_subarray(nums):
    # TODO: như Cách 2 nhưng lưu (giá trị, L+1, R-1) lớn nhất
    pass

def sum_subarray_mins(arr):
    # TODO: left dùng '>' (nghiêm ngặt) để pop, right dùng '>=' ... đóng góp arr[i] * (i - L) * (R - i)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: mỗi phần tử làm min. Mức 2: đoạn tối đa. Mức 3: (a) `1`; (b) `2 · 4 = 8`; (c) `5 · 5 = 25`; (d) `4 · 12 = 48` (`[4,4,4]` bên phải).

**Bài 2.** Mức 1: `L = [−1, 0, 0, −1, 3]`. Mức 2: `R = [3, 2, 3, 5, 5]`. Mức 3: min-product: `2·11 = 22`, `5·5 = 25`, `4·9 = 36`, `1·15 = 15`, `3·3 = 9` ⇒ `36`.

**Bài 3.** Mức 1: duyệt từ phải. Mức 2: pop `≥`. Mức 3: so với quét tuyến tính.

**Bài 4.** Mức 1: lưu chỉ số đạt max. Mức 2: đoạn `[L+1, R−1]`. Mức 3: kiểm tra `min(seg) * sum(seg)` bằng giá trị.

**Bài 5.** Mức 1: mỗi mảng con được tính cho **đúng một** chỉ số min (chọn bản đầu tiên hoặc cuối cùng khi bằng nhau). Mức 2: quy ước `L` = phần tử `<` gần nhất bên trái (pop khi `≥`), `R` = phần tử `≤` gần nhất bên phải (pop khi `>`), để mỗi mảng con chỉ gán cho bản sao min **ngoài cùng bên phải**. Mức 3: đóng góp `arr[i]·(i − L)·(R − i)`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2, 3
f = Solution().maxSumMinProduct
b1 = [f(a) for a in [[1], [2, 2], [1, 5, 1], [4, 4, 1, 4, 4, 4]]]
print("Bài 1:", b1); assert b1 == [1, 8, 25, 48]
assert f([2, 5, 4, 1, 3]) == 36 and prev_smaller([2, 5, 4, 1, 3]) == [-1, 0, 0, -1, 3]
def next_smaller_todo(nums):
    n = len(nums); right = [n] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] >= nums[i]: st.pop()
        if st: right[i] = st[-1]
        st.append(i)
    return right
print("Bài 2: R =", next_smaller_todo([2, 5, 4, 1, 3])); assert next_smaller_todo([2, 5, 4, 1, 3]) == [3, 2, 3, 5, 5]
print("Bài 3: OK")

Bài 1: [1, 8, 25, 48]
Bài 2: R = [3, 2, 3, 5, 5]
Bài 3: OK


In [28]:
# Bài 4, 5
def best_subarray(nums):
    n = len(nums); L = prev_smaller(nums); R = next_smaller_todo(nums); pre = list(accumulate(nums, initial=0))
    return max((nums[i] * (pre[R[i]] - pre[L[i] + 1]), L[i] + 1, R[i] - 1) for i in range(n))

def sum_subarray_mins(arr):
    n = len(arr); left, right, st = [-1] * n, [n] * n, []
    for i, x in enumerate(arr):
        while st and arr[st[-1]] >= x: st.pop()          # left: phần tử < x gần nhất
        left[i] = st[-1] if st else -1; st.append(i)
    st = []
    for i in range(n - 1, -1, -1):
        while st and arr[st[-1]] > arr[i]: st.pop()      # right: phần tử <= arr[i] gần nhất
        right[i] = st[-1] if st else n; st.append(i)
    return sum(arr[i] * (i - left[i]) * (right[i] - i) for i in range(n)) % MOD

rng = random.Random(5)
for _ in range(400):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 12))]
    v, l, r = best_subarray(a); seg = a[l:r + 1]
    assert v == min(seg) * sum(seg) and v % MOD == minprod_bruteforce(a)
    assert sum_subarray_mins(a) == sum(min(a[i:j]) for i in range(len(a)) for j in range(i + 1, len(a) + 1)) % MOD
print("Bài 4, 5: OK |", best_subarray([3, 1, 5, 6, 4, 2]), sum_subarray_mins([3, 1, 2, 4]), "(907 cho 17)")

Bài 4, 5: OK | (60, 2, 4) 17 (907 cho 17)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "mỗi phần tử làm min + stack đơn điệu":**

```python
left  = phần tử < nums[i] gần nhất bên trái  (stack tăng, pop khi >=)
right = phần tử < nums[i] gần nhất bên phải  (duyệt ngược, pop khi >=)
pre = tổng tiền tố
best = max(nums[i] * (pre[right[i]] - pre[left[i] + 1]))
return best % MOD          # mod SAU khi lấy max
```

**Checklist:**
- [ ] Mở rộng **cả hai phía** tới phần tử nhỏ hơn?
- [ ] Stack pop đúng chiều (`≥ x` ⇒ còn lại `< x`)?
- [ ] Tổng đoạn bằng **tiền tố** (không `sum` lặp)?
- [ ] Modulo **sau** khi lấy max?
- [ ] Đã test một phần tử, toàn bằng nhau, giá trị `10⁷` × `10⁵` phần tử?